In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

BASE_DIR = Path(
    '/content/drive/MyDrive/CBNU/어프렌티스 프로젝트/SKAB_Project'
)

IMG_DIR = Path(
    '/content/drive/MyDrive/CBNU/어프렌티스 프로젝트/'
    'SKAB_Project/img'
)

IMG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = BASE_DIR / 'data'
RESULT_DIR = BASE_DIR / 'results'

BASE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("프로젝트 경로:", BASE_DIR)
print("이미지 경로:", IMG_DIR)

In [ ]:
import subprocess
import shutil

REPO_DIR = BASE_DIR / 'SKAB'

if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/waico/SKAB.git',
        str(REPO_DIR)
    ], check=True)

DATA_DIR = REPO_DIR / 'data'

csv_files = sorted(DATA_DIR.rglob('*.csv'))

print("CSV 파일 개수:", len(csv_files))
for file in csv_files:
    print(file.relative_to(DATA_DIR))

In [ ]:
import pandas as pd

normal_path = DATA_DIR / 'anomaly-free' / 'anomaly-free.csv'

df = pd.read_csv(normal_path, sep=';')

print("데이터 크기:", df.shape)
print("컬럼 목록:", df.columns.tolist())

display(df.head())

In [ ]:
summary = []

for path in csv_files:
    df = pd.read_csv(path, sep=';')

    summary.append({
        'file': str(path.relative_to(DATA_DIR)),
        'rows': len(df),
        'columns': len(df.columns),
        'anomaly_count': (
            int(df['anomaly'].sum())
            if 'anomaly' in df.columns else None
        )
    })

summary_df = pd.DataFrame(summary)

display(summary_df)
print("전체 데이터 행 수:", summary_df['rows'].sum())

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# SKAB 데이터 경로
DATA_DIR = Path(
    '/content/drive/MyDrive/CBNU/어프렌티스 프로젝트/'
    'SKAB_Project/SKAB/data'
)

# 분석할 파일 선택
csv_files = sorted(DATA_DIR.rglob('*.csv'))

for i, path in enumerate(csv_files):
    print(f"{i}: {path.relative_to(DATA_DIR)}")

# 우선 첫 번째 파일 분석
file_path = csv_files[0]
df = pd.read_csv(file_path, sep=';')

df['datetime'] = pd.to_datetime(df['datetime'])

sensor_cols = [
    'Accelerometer1RMS',
    'Accelerometer2RMS',
    'Current',
    'Pressure',
    'Temperature',
    'Thermocouple',
    'Voltage',
    'Volume Flow RateRMS'
]

# 센서별 개별 그래프
for sensor in sensor_cols:
    fig, ax = plt.subplots(figsize=(14, 3.5))

    ax.plot(
        df['datetime'],
        df[sensor],
        linewidth=1,
        label=sensor
    )

    # 실제 이상 데이터 위치 표시
    if 'anomaly' in df.columns:
        mask = df['anomaly'] == 1

        ax.scatter(
            df.loc[mask, 'datetime'],
            df.loc[mask, sensor],
            color='red',
            s=9,
            label='Anomaly',
            zorder=3
        )

    ax.set_title(f"{file_path.name} - {sensor}")
    ax.set_xlabel('Time')
    ax.set_ylabel(sensor)
    ax.legend()
    ax.grid(alpha=0.3)

    plt.xticks(rotation=30)
    plt.tight_layout()

    # 이미지 파일 저장
    save_name = f"{file_path.stem}_{sensor}.png"
    save_name = save_name.replace(" ", "_")

    save_path = IMG_DIR / save_name

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()
    plt.close(fig)

    print(f"저장 완료: {save_path}")


In [ ]:
print(df.columns.tolist())

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path(
    '/content/drive/MyDrive/CBNU/어프렌티스 프로젝트/SKAB_Project'
)
DATA_DIR = BASE_DIR / 'SKAB/data'
IMG_DIR = BASE_DIR / 'img'
RESULT_DIR = BASE_DIR / 'results'

IMG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

SENSORS = [
    'Accelerometer1RMS',
    'Accelerometer2RMS',
    'Current',
    'Pressure',
    'Temperature',
    'Thermocouple',
    'Voltage',
    'Volume Flow RateRMS'
]

csv_files = sorted(DATA_DIR.rglob('*.csv'))

quality_results = []
sensor_results = []
label_results = []

for path in csv_files:
    df = pd.read_csv(path, sep=';')
    df.columns = df.columns.str.strip()

    file_name = str(path.relative_to(DATA_DIR))

    # 1. 구조 및 결측치 검사
    missing_cols = [c for c in SENSORS if c not in df.columns]
    missing_count = int(df.isna().sum().sum())
    duplicate_rows = int(df.duplicated().sum())

    # 2. 시간 데이터 검사
    duplicate_time = np.nan
    invalid_time = np.nan
    nonpositive_intervals = np.nan
    median_interval = np.nan

    if 'datetime' in df.columns:
        dt = pd.to_datetime(df['datetime'], errors='coerce')
        invalid_time = int(dt.isna().sum())
        duplicate_time = int(dt.dropna().duplicated().sum())

        diffs = dt.diff().dt.total_seconds().dropna()
        nonpositive_intervals = int((diffs <= 0).sum())

        positive_diffs = diffs[diffs > 0]
        if len(positive_diffs):
            median_interval = positive_diffs.median()

    # 3. 센서별 통계 및 IQR 검사
    for sensor in SENSORS:
        if sensor not in df.columns:
            continue

        values = pd.to_numeric(df[sensor], errors='coerce')
        values = values.replace([np.inf, -np.inf], np.nan)

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outlier_count = int(
            ((values < lower) | (values > upper)).sum()
        )

        sensor_results.append({
            'file': file_name,
            'sensor': sensor,
            'mean': values.mean(),
            'std': values.std(),
            'min': values.min(),
            'max': values.max(),
            'iqr_outliers': outlier_count
        })

    # 4. 이상 라벨 검사
    for label in ['anomaly', 'changepoint']:
        if label in df.columns:
            counts = df[label].value_counts(dropna=False)
            for value, count in counts.items():
                label_results.append({
                    'file': file_name,
                    'label': label,
                    'value': str(value),
                    'count': int(count)
                })

    quality_results.append({
        'file': file_name,
        'rows': len(df),
        'columns': len(df.columns),
        'missing_values': missing_count,
        'missing_sensors': ', '.join(missing_cols),
        'duplicate_rows': duplicate_rows,
        'duplicate_datetime': duplicate_time,
        'invalid_datetime': invalid_time,
        'nonpositive_intervals': nonpositive_intervals,
        'median_interval_sec': median_interval,
        'has_anomaly_label': 'anomaly' in df.columns
    })

quality_df = pd.DataFrame(quality_results)
sensor_df = pd.DataFrame(sensor_results)
label_df = pd.DataFrame(label_results)

display(quality_df)

print("CSV 파일 수:", len(csv_files))
print("전체 결측치:", quality_df['missing_values'].sum())
print("전체 중복 행:", quality_df['duplicate_rows'].sum())
print("전체 데이터 행 수:", quality_df['rows'].sum())

# 결과 저장
quality_df.to_csv(
    RESULT_DIR / 'data_quality_summary.csv',
    index=False, encoding='utf-8-sig'
)

sensor_df.to_csv(
    RESULT_DIR / 'sensor_statistics.csv',
    index=False, encoding='utf-8-sig'
)

label_df.to_csv(
    RESULT_DIR / 'label_distribution.csv',
    index=False, encoding='utf-8-sig'
)

print("품질검사 결과 저장 완료")


In [ ]:
label_summary = []

for path in csv_files:
    df = pd.read_csv(path, sep=';')
    df.columns = df.columns.str.strip()

    if 'anomaly' in df.columns:
        counts = df['anomaly'].value_counts()

        label_summary.append({
            'file': str(path.relative_to(DATA_DIR)),
            'normal': counts.get(0, 0),
            'anomaly': counts.get(1, 0),
            'anomaly_ratio': round(
                (df['anomaly'] == 1).mean() * 100, 2
            )
        })

label_summary_df = pd.DataFrame(label_summary)
display(label_summary_df)

In [ ]:
normal_df = pd.read_csv(
    DATA_DIR / 'anomaly-free/anomaly-free.csv',
    sep=';'
)

normal_df['datetime'] = pd.to_datetime(normal_df['datetime'])

print("데이터 행 수:", len(normal_df))
print("시작 시간:", normal_df['datetime'].min())
print("종료 시간:", normal_df['datetime'].max())
print("측정 시간 간격:")
print(normal_df['datetime'].diff().value_counts().head())

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

start = pd.Timestamp('2020-02-08 16:05:00')
end = pd.Timestamp('2020-02-08 16:15:00')

# 실제 데이터 날짜를 자동으로 사용
date = normal_df['datetime'].dt.normalize().iloc[0]
start = date + pd.Timedelta(hours=16, minutes=5)
end = date + pd.Timedelta(hours=16, minutes=15)

subset = normal_df[
    (normal_df['datetime'] >= start) &
    (normal_df['datetime'] <= end)
]

fig, axes = plt.subplots(
    len(SENSORS), 1,
    figsize=(15, 18),
    sharex=True
)

for ax, sensor in zip(axes, SENSORS):
    ax.plot(
        subset['datetime'],
        subset[sensor],
        linewidth=1
    )
    ax.set_ylabel(sensor, fontsize=9)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Time')
fig.suptitle('SKAB Normal Data - Sensor Comparison')
plt.tight_layout()

plt.savefig(
    IMG_DIR / 'normal_sensor_sudden_drop_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


In [ ]:
import numpy as np
import pandas as pd

cols = ['Accelerometer1RMS', 'Accelerometer2RMS']

check_df = normal_df[['datetime'] + cols].copy()

for col in cols:
    check_df[f'{col}_change'] = check_df[col].pct_change() * 100

# 두 센서가 동시에 3% 이상 하락한 시점
mask = (
    (check_df['Accelerometer1RMS_change'] < -3) &
    (check_df['Accelerometer2RMS_change'] < -3)
)

display(check_df.loc[mask])

In [ ]:
import pandas as pd

time_summary = []

for path in csv_files:
    df = pd.read_csv(path, sep=';')
    df.columns = df.columns.str.strip()

    dt = pd.to_datetime(df['datetime'])

    time_summary.append({
        'file': str(path.relative_to(DATA_DIR)),
        'rows': len(df),
        'start_time': dt.min(),
        'end_time': dt.max(),
        'duration_min': round(
            (dt.max() - dt.min()).total_seconds() / 60, 2
        ),
        'median_interval_sec': dt.sort_values()
            .diff().dt.total_seconds().median()
    })

time_df = pd.DataFrame(time_summary)

display(time_df)

time_df.to_csv(
    RESULT_DIR / 'skab_time_summary.csv',
    index=False,
    encoding='utf-8-sig'
)